In [13]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
print(p)
q = df[:, 6]

err_p = df[:, 8]

misurato = ~np.isnan(df[:, 2])     
err_p = df[:, 8]
err_q = np.where(misurato, df[:, 7], np.sqrt(df[:, 7]**2 ))

p = np.delete(p, [0] )
q = np.delete(q, [0]  )
err_p = np.delete(err_p, [0] )
err_q = np.delete(err_q, [0] )


#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

[ 232.5  233.5  234.5  235.5  236.5  237.5  238.5  239.5  240.5  241.5
  242.5  243.5  244.5  245.5  246.5  247.5  248.5  249.5  250.5  251.5
  252.5  253.5  254.5  255.5  256.5  257.5  258.5  259.5  260.5  261.5
  262.5  263.5  264.5  265.5  267.5  269.5  271.5  273.5  275.5  277.5
  280.5  281.5  282.5  283.5  284.5  285.5  286.5  287.5  288.5  289.5
  290.5  291.5  292.5  293.5  294.5  295.5  297.5  299.5  301.5  306.5
  311.5  316.5  321.5  326.5  331.5  336.5  341.5  346.5  351.5  356.5
  361.5  366.5  371.5  376.5  381.5  386.5  391.5  396.5  401.5  406.5
  411.5  416.5  421.5  426.5  431.5  441.5  451.5  461.5  471.5  481.5
  491.5  501.5  511.5  521.5  531.5  541.5  551.5  561.5  571.5  581.5
  601.5  621.5  641.5  661.5  681.5  731.5  781.5  831.5  881.5  981.5
 1081.5 1181.5 1281.5 1381.5 1481.5 1631.5]
Fit status: 0
Parametri stimati:
A = -0.07088868908219546 ± 0.21197252051686757
B = 2.6048360049547488 ± 0.15409678543070032
C = 200.2851262562941 ± 0.09362269578021043
Chi2 =

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [14]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [15]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff


#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 60.13024036823533 ± 8.011707813500173
#mu = 2.9521367300765418 ± 0.2522970863317493
#sigma = 2.1546321614604733 ± 0.16597667474227162
Chi2 = 34.80247433279619


-13.806817298066415
61
-7.033619657142348
68
-2.848901866800749
69
-5.5427618107208945
71
-12.950221509417487
72
-22.862704352976674
73
-29.613214131708478
74
-50.17905265060119
75
-61.52541794152822
76
-13.412553840323621
77
-58.98218809444384
87
-7.9400090100115275
93
-0.813859992270697
98
-9.362047663050692
100
-3.3767687244275306
104
-7.733653709594894
105
-3.735360179041861
106
-10.661164609521432
108
-12.174787767172418
109
-6.654970008373999
110
-7.340755178550644
112
-1.7114401202584482
113
-7.936493066210436
114
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      34.8025
NDf                       =           10
Edm                       =  1.18527e-06
NCalls                    =          181
const                     =      60.

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [16]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [17]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[ 1.14503271e+02  1.41740424e+02  1.08580156e+02  1.08140610e+02
  1.03844843e+02  1.01482358e+02  9.21667407e+01  9.35976529e+01
  8.20813709e+01  7.79030424e+01  8.68683576e+01  6.92113696e+01
  7.13681654e+01  6.34975970e+01  7.47980259e+01  7.30177492e+01
  6.92871927e+01  6.34910799e+01  6.26849858e+01  5.66790159e+01
  5.90561342e+01  5.52762181e+01  5.77539088e+01  5.17011305e+01
  4.86297679e+01  4.47293841e+01  4.17942983e+01  4.50670093e+01
  4.48902000e+01  4.09379580e+01  4.46361057e+01  4.04947805e+01
  3.89869037e+01  3.68472159e+01  3.55613064e+01  3.37770655e+01
  3.36531636e+01  3.16721399e+01  3.25107071e+01  2.73439676e+01
  2.89823695e+01  2.74868185e+01  2.19208519e+01  1.96140358e+01
  2.41944526e+01  2.11612166e+01  2.20092486e+01  2.37021869e+01
  2.12952721e+01  2.12713002e+01  2.31291711e+01  1.89265661e+01
  2.10635373e+01  1.96386502e+01  2.11036298e+01  1.97434434e+01
  1.95606876e+01  1.60268177e+01  1.56657577e+01  1.51383255e+01
  1.14835920e+00 -4.63061

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created


In [18]:
# residui normalizzati divisi perzona di p

limiti = [232, 260, 316, 340, 460, 700, 2000]

print("zona di p      N   media   errore sulla media   std")
for i in range(len(limiti) - 1):
    sel = (p >= limiti[i]) & (p < limiti[i+1]) 
    r = residuals_norm[sel]
    print(limiti[i], "-", limiti[i+1], "  ", len(r), "  ", round(np.mean(r), 2), "  ",
          round(np.std(r) / np.sqrt(len(r)), 2), "  ", round(np.std(r), 2))

r = residuals_norm[:8]
print("primi 8 punti:", len(r), "  ", round(np.mean(r), 2), "  ",
      round(np.std(r) / np.sqrt(len(r)), 2), "  ", round(np.std(r), 2))

#  punti con residuo oltre 2 sigma
print("punti con |residuo| > 2:")
for i in range(len(p)):
    if abs(residuals_norm[i]) > 2:
        print("  indice", i, " p =", p[i], " residuo normalizzato =", round(residuals_norm[i], 1),
              " residuo [mm] =", round(q[i] - q_attesi[i], 1))



zona di p      N   media   errore sulla media   std
232 - 260    27    3.36    0.1    0.5
260 - 316    33    4.26    0.16    0.92
316 - 340    5    3.18    4.01    8.97
340 - 460    21    -3.58    4.71    21.61
460 - 700    18    5.08    4.24    18.0
700 - 2000    11    -5.39    1.27    4.22
primi 8 punti: 8    2.89    0.11    0.31
punti con |residuo| > 2:
  indice 0  p = 233.5  residuo normalizzato = 3.4  residuo [mm] = 114.5
  indice 1  p = 234.5  residuo normalizzato = 3.0  residuo [mm] = 141.7
  indice 2  p = 235.5  residuo normalizzato = 2.5  residuo [mm] = 108.6
  indice 3  p = 236.5  residuo normalizzato = 2.7  residuo [mm] = 108.1
  indice 4  p = 237.5  residuo normalizzato = 2.6  residuo [mm] = 103.8
  indice 5  p = 238.5  residuo normalizzato = 2.6  residuo [mm] = 101.5
  indice 6  p = 239.5  residuo normalizzato = 3.1  residuo [mm] = 92.2
  indice 7  p = 240.5  residuo normalizzato = 3.3  residuo [mm] = 93.6
  indice 8  p = 241.5  residuo normalizzato = 2.7  residuo [mm] = 8